# Pruned SAM — LoRA + 知识蒸馏微调

在 Colab T4 GPU 上对剪枝后的 Pruned-M 模型进行微调，恢复分割精度。

**预计时间**: ~15 分钟（含数据下载、预计算 Teacher 标签、30 epochs 训练）

**输出**: `pruned_m_ft_final.pth`（完整微调模型）+ `pruned_m_lora_only.pth`（轻量 LoRA 权重）

In [ ]:
# @title 1. 检查 GPU 并安装依赖
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!pip install -q pycocotools tqdm

In [ ]:
# @title 2. 克隆代码仓库
import os
GIT_REPO = "https://github.com/buaa-zy-2239/pruned-sam.git"

if not os.path.exists('/content/vista-slam'):
    !git clone {GIT_REPO} /content/vista-slam
else:
    print("Already cloned")

%cd /content/vista-slam

In [ ]:
# @title 3. 下载权重和评估数据
import gdown
import os
import zipfile

# Google Drive 文件 ID（需要你上传后替换）
# 或者直接从你的本地机器上传到 Colab
from google.colab import files

print("请上传 vista-slam-data.zip（包含 weights + eval_data）")
print("如果还没有压缩包，请先在本机执行: ")
print("  cd /home/zhang/vista-slam")
print("  tar czf vista-slam-data.tar.gz \\")
print("    pruned_sam/weights/pruned_m.pth \\")
print("    pruned_sam/weights/pruned_l.pth \\")
print("    pruned_sam/weights/swr_gating.pth \\")
print("    TinySAM/weights/tinysam_42.3.pth \\")
print("    eval_data/test_100/ eval_data/partial_annotations/")

# 或者使用 gdown 从 Google Drive 下载（推荐，更稳定）
USE_GDRIVE = False  # 改为 True 并使用你的文件ID
GDRIVE_FILE_ID = "YOUR_FILE_ID_HERE"

if USE_GDRIVE and GDRIVE_FILE_ID != "YOUR_FILE_ID_HERE":
    !gdown --id {GDRIVE_FILE_ID} -O /content/vista-slam-data.tar.gz
    !tar xzf /content/vista-slam-data.tar.gz -C /content/vista-slam
    print("数据解压完成!")
else:
    print("\n请手动上传文件...")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.tar.gz'):
            !tar xzf {fn} -C /content/vista-slam
            print(f"解压 {fn} 完成!")
        elif fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zf:
                zf.extractall('/content/vista-slam')
            print(f"解压 {fn} 完成!")

In [ ]:
# @title 4. 验证文件完整性
import os

files_to_check = [
    'TinySAM/weights/tinysam_42.3.pth',
    'pruned_sam/weights/pruned_m.pth',
    'eval_data/partial_annotations/instances_val2017.json',
]

all_ok = True
for f in files_to_check:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1e6
        print(f"  ✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {f} 缺失!")
        all_ok = False

# 检查 test_100 图片数量
img_dir = 'eval_data/test_100'
if os.path.exists(img_dir):
    n_imgs = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    print(f"  ✅ {img_dir}: {n_imgs} 张图片")

if all_ok:
    print("\n✅ 所有文件完整，可以开始训练!")
else:
    print("\n❌ 请上传缺失的文件")

In [ ]:
# @title 5. 开始微调
!python pruned_sam/train_lora_distill.py

In [ ]:
# @title 6. 下载训练好的模型
from google.colab import files
import os

weights_dir = 'pruned_sam/weights'
for f in os.listdir(weights_dir):
    if f.endswith('.pth') and ('ft_' in f or 'lora_' in f):
        path = os.path.join(weights_dir, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f"下载 {f} ({size_mb:.1f} MB)...")
        files.download(path)

In [ ]:
# @title (可选) 微调后评估
!python pruned_sam/evaluate_box_miou.py